# SparseVol3D — Google Colab Setup

**One-click setup for running SparseVol3D on Google Colab (free GPU)**

### Before running:
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Then click **Runtime → Run all**

---
**What this notebook does:**
- Installs all dependencies
- Clones the SparseVol3D repo from GitHub
- Downloads KiTS19 data (or generates synthetic data for a quick test)
- Trains the model
- Evaluates and plots results

## Step 1 — Check GPU

In [ ]:
import torch
print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU             :', torch.cuda.get_device_name(0))
    print('VRAM            :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected.')
    print('Go to Runtime > Change runtime type > T4 GPU')

## Step 2 — Install dependencies & clone repo

In [ ]:
# Install required packages
!pip install nibabel tqdm scikit-image scipy -q

# Clone the SparseVol3D repo
import os
if not os.path.exists('SparseVol3D'):
    !git clone https://github.com/NatalieCarlebach1/SparseVol3D.git

os.chdir('SparseVol3D')
print('Working directory:', os.getcwd())

import sys
sys.path.insert(0, os.getcwd())

## Step 3 — Data

**Option A** (recommended for real results): download KiTS19 (~50 GB, ~1 hour)

**Option B** (quick test, ~10 seconds): use synthetic data

Set `USE_SYNTHETIC = True` for a quick smoke test, or `False` to download the real dataset.

In [ ]:
USE_SYNTHETIC = True   # <-- change to False to download real KiTS19 data

DATA_DIR = 'data/kits19/data'
os.makedirs(DATA_DIR, exist_ok=True)

if USE_SYNTHETIC:
    print('Generating 15 synthetic cases...')
    !python scripts/make_synthetic_data.py --n_cases 15 --data_dir {DATA_DIR}
else:
    print('Downloading KiTS19 (~50 GB)...')
    if not os.path.exists('../kits19'):
        !git clone https://github.com/neheller/kits19.git ../kits19
    !cd ../kits19 && pip install -r requirements.txt -q
    !cd ../kits19 && python starter_code/get_imaging.py
    if not os.path.exists(DATA_DIR):
        os.symlink('../kits19/data', DATA_DIR)

import glob
cases = sorted(glob.glob(os.path.join(DATA_DIR, 'case_*')))
print(f'\nReady: {len(cases)} cases in {DATA_DIR}')

In [ ]:
import json

!python scripts/prepare_splits.py --data_dir {DATA_DIR}

with open('outputs/splits.json') as f:
    splits = json.load(f)
print(f"Train: {len(splits['train'])}  Val: {len(splits['val'])}  Test: {len(splits['test'])}")

## Step 4 — Configure & Train

Change `LABEL_STRIDE` and `LAMBDA_VIC` to run different experiments.

In [ ]:
LABEL_STRIDE = 5      # 1=dense (100%), 5=every 5th (20%), 10=every 10th (10%)
LAMBDA_VIC   = 0.1    # 0.0 = no VIC loss (ablation), 0.1 = SparseVol3D
EPOCHS       = 100
OUTPUT_DIR   = f'outputs/colab_s{LABEL_STRIDE}_vic{LAMBDA_VIC}'

print(f'label_stride = {LABEL_STRIDE}  ({100 // LABEL_STRIDE}% annotation)')
print(f'lambda_vic   = {LAMBDA_VIC}')

In [ ]:
!python train.py \
    --data_dir     {DATA_DIR} \
    --output_dir   {OUTPUT_DIR} \
    --label_stride {LABEL_STRIDE} \
    --lambda_vic   {LAMBDA_VIC} \
    --epochs       {EPOCHS}

## Step 5 — Evaluate

In [ ]:
!python evaluate.py \
    --checkpoint  {OUTPUT_DIR}/best_model.pt \
    --data_dir    {DATA_DIR} \
    --splits_file outputs/splits.json \
    --split       val

## Step 6 — Plot Results

In [ ]:
import json, os
import numpy as np
import matplotlib.pyplot as plt

with open(f'{OUTPUT_DIR}/train_log.json') as f:
    log = json.load(f)

epochs_with_dice = [r for r in log if 'kidney_dice' in r]
ep = [r['epoch']       for r in epochs_with_dice]
kd = [r['kidney_dice'] for r in epochs_with_dice]
td = [r['tumor_dice']  for r in epochs_with_dice]
ls = [r['train_loss']  for r in log]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(range(1, len(ls)+1), ls, color='steelblue', lw=2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training Loss'); ax1.grid(alpha=0.3)

ax2.plot(ep, kd, label='Kidney', color='royalblue', lw=2)
ax2.plot(ep, td, label='Tumor',  color='tomato',    lw=2)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Dice')
ax2.set_title(f'Validation Dice (stride={LABEL_STRIDE}, VIC={LAMBDA_VIC})')
ax2.legend(); ax2.grid(alpha=0.3); ax2.set_ylim(0, 1)

plt.tight_layout(); plt.show()

print(f'Best Kidney Dice : {max(kd):.4f}')
print(f'Best Tumor  Dice : {max(td):.4f}')
print(f'Best Mean   Dice : {(max(kd)+max(td))/2:.4f}')

## Step 7 — Save to Google Drive (optional)

Mount your Drive to keep the checkpoint after the Colab session ends.

In [ ]:
SAVE_TO_DRIVE = False   # set True to save checkpoint to Google Drive

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil
    dest = f'/content/drive/MyDrive/SparseVol3D/{os.path.basename(OUTPUT_DIR)}'
    shutil.copytree(OUTPUT_DIR, dest, dirs_exist_ok=True)
    print(f'Saved to {dest}')